# 03 - Data Cleaning
## London Safety Analysis - Kudzanayi Shepherd Mhlanga

This notebook cleans the raw crime data and produces a tidy, analysis-ready dataset in `data/processed/`.

**Cleaning steps:**
1. Remove duplicates
2. Handle missing values
3. Standardise borough and crime-type names
4. Filter out invalid coordinates
5. Add derived date columns
6. Merge population data
7. Save cleaned output

---

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))
from src.config import DATA_RAW, DATA_PROCESSED, FIGURES_DIR, LONDON_BOROUGHS

plt.style.use("seaborn-v0_8-whitegrid")

raw = pd.read_csv(DATA_RAW / "crime_data_raw.csv", parse_dates=["month"])
print(f"Raw data loaded: {len(raw):,} rows")
raw.head(3)


## 1. Inspect raw data quality

In [ ]:
print("Shape:", raw.shape)
print()
print("Dtypes:")
print(raw.dtypes)
print()
print("Sample nulls and empty strings:")
for col in raw.columns:
    n_null  = raw[col].isnull().sum()
    n_empty = (raw[col].astype(str).str.strip() == "").sum()
    if n_null + n_empty > 0:
        print(f"  {col:<25}  nulls={n_null:,}  empty_strings={n_empty:,}")


## 2. Remove duplicates

In [ ]:
before = len(raw)
df = raw.drop_duplicates(subset=["crime_id"])
removed = before - len(df)
print(f"Rows before  : {before:,}")
print(f"Duplicates   : {removed:,}")
print(f"Rows after   : {len(df):,}")


## 3. Handle missing values

In [ ]:
# Coordinates: drop rows where lat/lon is exactly 0 or NaN
df["latitude"]  = pd.to_numeric(df["latitude"],  errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

invalid_coords = (
    df["latitude"].isna()  | df["longitude"].isna() |
    (df["latitude"] == 0)  | (df["longitude"] == 0)
)
print(f"Records with invalid coordinates: {invalid_coords.sum():,}")
df = df[~invalid_coords].copy()

# Outcome category: fill blank with "Not recorded"
df["outcome_category"] = df["outcome_category"].fillna("Not recorded")
df.loc[df["outcome_category"].str.strip() == "", "outcome_category"] = "Not recorded"

# Street name: fill blank with "Unknown street"
df["street_name"] = df["street_name"].fillna("Unknown street")
df.loc[df["street_name"].str.strip() == "", "street_name"] = "Unknown street"

print(f"Rows remaining: {len(df):,}")
print("Remaining nulls:")
print(df.isnull().sum()[df.isnull().sum() > 0])


## 4. Standardise borough names

In [ ]:
# Check for any unexpected borough values
all_boroughs_in_data = set(df["borough"].unique())
expected = set(LONDON_BOROUGHS)
unexpected = all_boroughs_in_data - expected
print(f"Unique boroughs in data : {len(all_boroughs_in_data)}")
print(f"Expected boroughs       : {len(expected)}")
if unexpected:
    print(f"Unexpected values       : {unexpected}")
else:
    print("All borough names match expected list. No standardisation needed.")

# Normalise capitalisation anyway
df["borough"] = df["borough"].str.strip().str.title()


## 5. Standardise crime types

In [ ]:
print("Crime types found in data:")
for ct in sorted(df["crime_type"].unique()):
    n = (df["crime_type"] == ct).sum()
    print(f"  {ct:<45} {n:,}")


In [ ]:
# Normalise to lowercase with hyphens (matches API convention)
df["crime_type"] = (
    df["crime_type"]
    .str.strip()
    .str.lower()
    .str.replace(" ", "-")
)

# Map any legacy or variant names
crime_type_map = {
    "other-crimes":                   "other-crime",
    "sexual-offences":                "violence-and-sexual-offences",
    "violence":                       "violence-and-sexual-offences",
    "asb":                            "anti-social-behaviour",
    "theft":                          "other-theft",
}
df["crime_type"] = df["crime_type"].replace(crime_type_map)
print("Crime type standardisation complete.")
print(f"Final unique crime types: {df['crime_type'].nunique()}")


## 6. Add derived date columns

In [ ]:
df["year"]        = df["month"].dt.year
df["month_num"]   = df["month"].dt.month
df["quarter"]     = df["month"].dt.quarter
df["month_label"] = df["month"].dt.strftime("%b %Y")

# Classify time period
df["period"] = df["year"].map({2024: "2024", 2025: "2025"})

print("Date columns added.")
df[["month","year","month_num","quarter","month_label","period"]].head(3)


## 7. Merge population data

In [ ]:
pop_path = DATA_RAW / "borough_population.csv"
if pop_path.exists():
    pop = pd.read_csv(pop_path)
    df = df.merge(pop, on="borough", how="left")
    print(f"Population data merged. Rows: {len(df):,}")
    missing_pop = df["population"].isna().sum()
    print(f"Missing population values: {missing_pop}")
else:
    print("Population file not found - skipping merge")
    df["population"] = None


## 8. Save cleaned data

In [ ]:
output_path = DATA_PROCESSED / "crime_data_clean.csv"
df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Shape: {df.shape}")
print()
print("Final column list:")
for col in df.columns:
    null_count = df[col].isna().sum()
    print(f"  {col:<25}  dtype={str(df[col].dtype):<12}  nulls={null_count}")


## 9. Cleaning summary

In [ ]:
print("=" * 55)
print("DATA CLEANING SUMMARY")
print("=" * 55)
print(f"  Raw records         : {len(raw):,}")
print(f"  After deduplication : {len(raw.drop_duplicates(subset=['crime_id'])):,}")
print(f"  After coord filter  : {len(df):,}")
print(f"  Records removed     : {len(raw) - len(df):,} ({(len(raw)-len(df))/len(raw)*100:.1f}%)")
print()
print(f"  Boroughs            : {df['borough'].nunique()}")
print(f"  Crime types         : {df['crime_type'].nunique()}")
print(f"  Date range          : {df['month'].min().strftime('%b %Y')} - {df['month'].max().strftime('%b %Y')}")
print()
print("Clean data saved to data/processed/crime_data_clean.csv")
print("Next step: Notebook 04 - Feature Engineering & Safety Scores")
